# Mushroom Edible vs Poisonous Classifier

Trains a binary image classifier (edible / poisonous) on the Kaggle dataset
"Over 3000 images of edible and poisonous sporocarps" and exports a
quantized TFLite model for on-device inference in Pocket Medic.

**Dataset**: https://www.kaggle.com/datasets/ [edible-poisonous-mushroom-sporocarps]
**Target classes**: `edible`, `poisonous`
**Model**: MobileNetV2 (transfer learning, backbone unfrozen during fine-tune)
**Output**: `mushroom_classifier.tflite` + `labels.txt`

In [ ]:
!pip install -q tensorflow matplotlib
import tensorflow as tf
print('TensorFlow:', tf.__version__)

## 1. Download and prepare the dataset

In [ ]:
import os, zipfile, shutil

DATA_DIR = '/content/mushroom_data'
EDIBLE_DIR = os.path.join(DATA_DIR, 'edible mushroom sporocarp')
POISON_DIR = os.path.join(DATA_DIR, 'poisonous mushroom sporocarp')

zip_path = '/content/mushroom_dataset.zip'
if os.path.exists(zip_path) and not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')

def count_images(folder):
    if not os.path.isdir(folder):
        return 0
    return sum(1 for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png')))

n_edible = count_images(EDIBLE_DIR)
n_poison = count_images(POISON_DIR)
print(f'Edible: {n_edible}  |  Poisonous: {n_poison}  |  Total: {n_edible + n_poison}')

## 2. Build tf.data pipelines with augmentation

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_HEAD = 15
EPOCHS_FINETUNE = 10

CONSOLIDATED = '/content/mushroom_consolidated'
os.makedirs(os.path.join(CONSOLIDATED, 'edible'), exist_ok=True)
os.makedirs(os.path.join(CONSOLIDATED, 'poisonous'), exist_ok=True)

def copy_images(src_dir, dst_subdir):
    os.makedirs(dst_subdir, exist_ok=True)
    for f in os.listdir(src_dir):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_subdir, f))

copy_images(EDIBLE_DIR, os.path.join(CONSOLIDATED, 'edible'))
copy_images(POISON_DIR, os.path.join(CONSOLIDATED, 'poisonous'))

train_ds = tf.keras.utils.image_dataset_from_directory(
    CONSOLIDATED, validation_split=0.2, subset='training', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='int',
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    CONSOLIDATED, validation_split=0.2, subset='validation', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='int',
)

class_names = train_ds.class_names
print('Classes:', class_names)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE,
).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

## 3. Build model (MobileNetV2 + classification head)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## 4. Train classification head

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
    ],
)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout()
plt.savefig('/content/training_head.png', dpi=150)
plt.show()

## 5. Fine-tune: unfreeze top 30 layers of MobileNetV2

In [ ]:
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINETUNE,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7),
    ],
)

best_acc = max(history_fine.history['val_accuracy'])
print(f'Best val accuracy: {best_acc:.4f}')
assert best_acc > 0.80, f'Accuracy {best_acc:.2%} is too low — check data or retrain'

## 6. Export TFLite — float16 quantization

Uses float16 for both weights and activations. This preserves the sigmoid output
precision (critical for binary classification) while still being 2x smaller than
float32. Input stays uint8 (fast on mobile).

In [ ]:
export_dir = '/content/output'
os.makedirs(export_dir, exist_ok=True)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset():
    for images, _ in val_ds.unbatch().batch(1).take(100):
        yield [images.numpy().astype('float32')]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

model_path = os.path.join(export_dir, 'mushroom_classifier.tflite')
with open(model_path, 'wb') as f:
    f.write(tflite_model)

print(f'Model size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB')

labels_path = os.path.join(export_dir, 'labels.txt')
with open(labels_path, 'w') as f:
    for name in class_names:
        f.write(name + '\n')

print('Exported:', model_path)
print('Labels:', labels_path)

## 7. Verify TFLite model

In [ ]:
import numpy as np

interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print('Input dtype:', input_details[0]['dtype'], 'shape:', input_details[0]['shape'])
print('Output dtype:', output_details[0]['dtype'], 'shape:', output_details[0]['shape'])

# Test with random image
dummy = np.random.randint(0, 255, (1, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
interpreter.set_tensor(input_details[0]['index'], dummy)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])
print('Raw output:', output)

# For sigmoid output: output < 0.5 = class 0, >= 0.5 = class 1
if output.shape[-1] == 1:
    prob = float(output[0][0])
    label = class_names[0] if prob < 0.5 else class_names[1]
    conf = prob if prob >= 0.5 else 1.0 - prob
    print(f'Predicted: {label} ({conf:.2%})')
else:
    idx = np.argmax(output)
    print(f'Predicted: {class_names[idx]} ({output[0][idx]:.2%})')

## 8. Download model files

Download `mushroom_classifier.tflite` and `labels.txt` to
your Flutter project's `assets/` directory, then run `flutter pub get`.

In [ ]:
from google.colab import files
files.download(model_path)
files.download(labels_path)